## Build Style Vectors — Llama-3.1-8B-Instruct

Place these files in the **same directory**:
```
main.py
build_vectors_llama31.ipynb
```

### Key differences from Llama-2 notebooks
| Setting | Llama-2-13b | Llama-2-7b | **Llama-3.1-8B** |
|---|---|---|---|
| `LOCAL_MODEL_NAME` | Llama-2-13b-chat-hf | Llama-2-7b-hf | **Llama-3.1-8B-Instruct** |
| `STEER_LAYER` | 28 | 20 | **20** |
| `STEER_ALPHA` | 12.0 | 15.0 | **13.0** |
| Hidden dim | 5120 | 4096 | **4096** |
| VRAM needed | ~26 GB | ~14 GB | **~16 GB** |
| Prompt format | `[INST]<<SYS>>` | `[INST]` | **`<\|begin_of_text\|>`** |
| EOS tokens | 1 | 1 | **2** (`<\|end_of_text\|>` + `<\|eot_id\|>`) |
| `add_special_tokens` | True | True | **False** |
| `transformers` | 4.40.0 | 4.40.0 | **≥4.43.0** |

### Runtime
**A100 (40 GB)** or **L4 (24 GB)**. Llama-3.1-8B in bfloat16 uses ~16 GB VRAM.

### Run order
**Cell 0a → Cell 0b → Cell 1 → … → Cell 11** (always start with 0a to clear previous models)


In [10]:
# ─── Cell 0a — AGGRESSIVE GPU CLEANUP (RUN THIS FIRST EVERY SESSION) ─────────
#
# Clears any previous model (e.g. leftover Llama-2 weights) before loading 3.1.
# If VRAM still shows > 1 GB after this cell, do:
#   Runtime → Restart runtime → re-run from Cell 0a.

import gc, sys

# Step 1: Wipe main_module model cache (the actual source of VRAM leaks)
if 'main_module' in dir():
    try:
        main_module._model_cache.clear()
        print('  ✓ main_module._model_cache.clear() — done')
    except Exception as e:
        print(f'  cache clear skipped: {e}')

# Step 2: Delete model / tokenizer from all globals
_gone = []
for _v in [k for k in list(globals().keys())
           if any(x in k.lower() for x in ['model', 'tokenizer', 'main_module'])]:
    try:
        del globals()[_v]
        _gone.append(_v)
    except Exception:
        pass
if _gone:
    print(f'  ✓ Deleted globals: {_gone}')

# Step 3: Evict main_module from sys.modules so Cell 5 reimports cleanly
_unloaded = [k for k in list(sys.modules.keys()) if 'main_module' in k]
for _m in _unloaded:
    del sys.modules[_m]
if _unloaded:
    print(f'  ✓ Unloaded from sys.modules: {_unloaded}')

# Step 4: GC + flush CUDA cache
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved  = torch.cuda.memory_reserved()  / 1e9
        total     = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'\n  GPU         : {torch.cuda.get_device_name(0)}')
        print(f'  Total VRAM  : {total:.1f} GB')
        print(f'  Allocated   : {allocated:.2f} GB  ← should be ~0.0 GB')
        print(f'  Reserved    : {reserved:.2f} GB')
        print(f'  Approx free : {total - reserved:.1f} GB')
        print()
        if allocated > 1.0:
            print('  ⚠  WARNING: VRAM still in use — a process is holding the GPU.')
            print('  → Runtime → Restart runtime, then re-run from Cell 0a.')
        elif total < 16.0:
            print(f'  ⚠  WARNING: Only {total:.1f} GB total VRAM.')
            print('  Llama-3.1-8B needs ~16 GB in bfloat16.')
            print('  → Runtime → Change runtime type → A100 (40 GB) or L4 (24 GB)')
        else:
            print('  ✓ VRAM clean and sufficient — safe to load Llama-3.1-8B-Instruct.')
    else:
        print('  ⚠  No GPU detected! → Runtime → Change runtime type → GPU')
except ImportError:
    print('  torch not installed yet — will be installed in Cell 1')


  ✓ main_module._model_cache.clear() — done
  ✓ Deleted globals: ['main_module', 'AutoModelForCausalLM', 'AutoTokenizer', 'MODEL_NAME', 'MODEL_NEEDS_GB', 'tokenizer']

  GPU         : NVIDIA RTX 6000 Ada Generation
  Total VRAM  : 50.9 GB
  Allocated   : 10.65 GB  ← should be ~0.0 GB
  Reserved    : 10.74 GB
  Approx free : 40.1 GB

  ⚠  WARNING: VRAM still in use — a process is holding the GPU.
  → Runtime → Restart runtime, then re-run from Cell 0a.


In [11]:
# ─── Cell 0b — MOUNT GOOGLE DRIVE FOR PERSISTENT MODEL CACHE ─────────────────
#
# FIXES: 'model re-downloads every session'
# Colab wipes /root/.cache on disconnect. Pointing HF_HOME to Drive means
# the ~16 GB weights download ONCE and reuse across all future sessions.
#
# First run : downloads ~16 GB to Drive (~5-8 min on A100)
# Later runs: loads from Drive cache (~1-2 min)
#
# Skip ONLY if you prefer the ephemeral /root/.cache (re-downloads every time).

import os

try:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_CACHE = '/content/drive/MyDrive/hf_model_cache'
    os.makedirs(DRIVE_CACHE, exist_ok=True)

    # Set ALL HuggingFace cache env vars BEFORE any transformers import
    os.environ['HF_HOME']               = DRIVE_CACHE
    os.environ['TRANSFORMERS_CACHE']    = DRIVE_CACHE
    os.environ['HF_DATASETS_CACHE']     = DRIVE_CACHE
    os.environ['HUGGINGFACE_HUB_CACHE'] = DRIVE_CACHE

    import glob
    existing = (glob.glob(f'{DRIVE_CACHE}/**/*.safetensors', recursive=True)
                + glob.glob(f'{DRIVE_CACHE}/**/*.bin', recursive=True))
    print(f'  ✓ HuggingFace cache → {DRIVE_CACHE}')
    if existing:
        total_gb = sum(os.path.getsize(f) for f in existing) / 1e9
        print(f'  ✓ Found {len(existing)} cached weight file(s) ({total_gb:.1f} GB) — NO re-download needed!')
    else:
        print('  ℹ  No cached weights yet — first run downloads ~16 GB to Drive.')
        print('  All future sessions will reuse this cache automatically.')
except Exception as e:
    print(f'  Drive mount skipped ({e}) — using /root/.cache (re-downloads every session).')


  Drive mount skipped (No module named 'google.colab') — using /root/.cache (re-downloads every session).


In [12]:
# ─── Cell 1 — Install dependencies ──────────────────────────────────────────
# Force-upgrade transformers FIRST before anything else installs/pins it
%pip install -q "transformers==4.43.3" "pydantic>=2.9.0"

%pip install -q \
    fastapi==0.115.0 \
    uvicorn==0.30.6 \
    httpx==0.27.2 \
    python-dotenv>=1.0.0 \
    langchain-groq>=0.1.9 \
    langchain-core>=0.2.0 \
    rich>=13.7.0 \
    accelerate>=0.29.0 \
    huggingface_hub>=0.22.0 \
    scikit-learn>=1.4.0 \
    numpy>=1.26.0 \
    spacy

try:
    import torch
    print(f"✓ torch already available: {torch.__version__}")
except ImportError:
    !pip install -q "torch>=2.2.0"
    import torch

!python -m spacy download en_core_web_sm

# ── Verify the right transformers version is active ──────────────────────────
import importlib, transformers
importlib.reload(transformers)
print(f"✓ transformers active version: {transformers.__version__}")
assert transformers.__version__ >= "4.43.0", \
    f"Still on {transformers.__version__} — RESTART KERNEL and rerun this cell."
print("✓ All dependencies installed")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
✓ torch already available: 2.10.0+cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 12.4 MB/s  0:00:01 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
✓ transformers active version: 4.43.3
✓ All dependencies installed


# after running restart the kernel and run again for libraries and error will go

In [ ]:
# ─── Cell 2 — Set secrets and authenticate with Hugging Face ─────────────────
#
# Option A (recommended): export HF_TOKEN and GROQ_API_KEY before launching.
# Option B: fill in the strings below (do NOT commit credentials to git).

import os
from huggingface_hub import login

# ── Uncomment and fill in if not already exported ────────────────────────────
os.environ["HF_TOKEN"]     = "HF_TOKEN"   # https://huggingface.co/settings/tokens
os.environ["GROQ_API_KEY"] = "GROQ_API_KEY"    # https://console.groq.com

HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise EnvironmentError(
        "HF_TOKEN not found.\n"
        "Run:  export HF_TOKEN='hf_...'\n"
        "Or uncomment the assignment line above."
    )

login(token=HF_TOKEN)
print("✓ Hugging Face login successful")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if GROQ_API_KEY:
    print("✓ GROQ_API_KEY found — full pipeline (Cell 12) will work")
else:
    print("⚠  GROQ_API_KEY not set — vector build/smoke-test will still work; Cell 12 requires it")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✓ Hugging Face login successful
✓ GROQ_API_KEY found — full pipeline (Cell 12) will work


In [14]:
# ─── Cell 3 — Verify GPU + library versions ───────────────────────────────────

import torch, transformers
print(f'torch        : {torch.__version__}')
print(f'transformers : {transformers.__version__}')
print(f'CUDA         : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU          : {gpu_name}')
    print(f'VRAM         : {vram_gb:.1f} GB')
    if vram_gb < 16.0:
        print('⚠  WARNING: Llama-3.1-8B-Instruct needs ~16 GB VRAM in bfloat16.')
        print('   Upgrade: Runtime → Change runtime type → A100 (40 GB) or L4 (24 GB)')
    else:
        print('✓ Sufficient VRAM for Llama-3.1-8B-Instruct in bfloat16')
else:
    print('⚠  No GPU detected → Runtime → Change runtime type → GPU')

# transformers version guard
from packaging import version as _v
if _v.parse(transformers.__version__) < _v.parse('4.43.0'):
    print(f'⚠  transformers {transformers.__version__} < 4.43.0 — Llama-3.1 may not work correctly!')
    print('   Run: pip install -q transformers==4.43.0  then restart kernel')
else:
    print(f'✓ transformers {transformers.__version__} >= 4.43.0 — Llama-3.1 fully supported')


torch        : 2.10.0+cu128
transformers : 4.43.3
CUDA         : True
GPU          : NVIDIA RTX 6000 Ada Generation
VRAM         : 50.9 GB
✓ Sufficient VRAM for Llama-3.1-8B-Instruct in bfloat16
✓ transformers 4.43.3 >= 4.43.0 — Llama-3.1 fully supported


In [15]:
# ─── Cell 4 — Verify main.py is the Llama-3.1 version ────────────────────────

from pathlib import Path

main_file = Path('main.py')
if not main_file.exists():
    raise FileNotFoundError(
        f'main.py not found in {Path.cwd()}.\n'
        'Upload main.py to the same directory as this notebook.'
    )

content = main_file.read_text()

checks = {
    'Llama-3.1-8B-Instruct in LOCAL_MODEL_NAME default': 'Llama-3.1-8B-Instruct' in content,
    'Llama-3.1 prompt format (begin_of_text)':           'begin_of_text'          in content,
    'Dual EOS token logic (eot_id)':                     'eot_id'                 in content,
    'add_special_tokens=False':                          'add_special_tokens=False' in content,
    '_model_cache dict present':                         '_model_cache'           in content,
}

all_ok = True
for desc, result in checks.items():
    icon = '✓' if result else '✗'
    print(f'  {icon} {desc}')
    if not result:
        all_ok = False

if not all_ok:
    raise ValueError(
        'main.py failed one or more checks above.\n'
        'Make sure you uploaded the Llama-3.1 version of main.py.'
    )

print(f'\n✓ main.py at {main_file.resolve()} — all checks passed')
print(f'  Size: {main_file.stat().st_size / 1024:.1f} KB')


  ✓ Llama-3.1-8B-Instruct in LOCAL_MODEL_NAME default
  ✓ Llama-3.1 prompt format (begin_of_text)
  ✓ Dual EOS token logic (eot_id)
  ✓ add_special_tokens=False
  ✓ _model_cache dict present

✓ main.py at /home/pakdd/github repo/EMNLP 2026/A2A_KG_Llama-3.1-8b-Instruct/main.py — all checks passed
  Size: 97.9 KB


In [16]:
# ─── Cell 5 — Configure environment and import main.py ────────────────────────
#
# These env vars must be set BEFORE importing main.py because module-level
# code in main.py reads them at import time.
#
# Llama-3.1-8B-Instruct settings:
#   STEER_LAYER = 20   (32 layers total; layer 20 ≈ 63% depth)
#   STEER_ALPHA = 13.0 (hidden=4096; slightly lower than Llama-2-7b's 15.0
#                       because 3.1 representations are more steerable)

import os, importlib.util
from pathlib import Path

os.environ['LOCAL_MODEL_NAME'] = 'meta-llama/Llama-3.1-8B-Instruct'
os.environ['STEER_LAYER']      = '20'
os.environ['STEER_ALPHA']      = '13.0'
os.environ['STYLE_CACHE_DIR']  = '.style_cache'
os.environ['HF_TOKEN']         = HF_TOKEN

Path('.style_cache').mkdir(exist_ok=True)

spec        = importlib.util.spec_from_file_location('main_module', 'main.py')
main_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(main_module)

print('✓ main.py imported successfully')
print('=' * 60)
print(f'  Model       : {os.environ["LOCAL_MODEL_NAME"]}')
print(f'  Steer layer : {os.environ["STEER_LAYER"]}')
print(f'  Steer alpha : {os.environ["STEER_ALPHA"]}')
print(f'  Cache dir   : {os.environ["STYLE_CACHE_DIR"]}')
print(f'  Contrast pairs: empathetic={len(main_module.CONTRAST_PAIRS["empathetic"])},'
      f' formal={len(main_module.CONTRAST_PAIRS["formal"])}')
print('=' * 60)

# Trigger spaCy load now so any missing-model errors surface early
nlp_check = main_module._get_nlp()
if nlp_check:
    print('✓ spaCy en_core_web_sm loaded for KG-NER')
else:
    print('⚠  spaCy unavailable — regex-only NER fallback will be active')


✓ main.py imported successfully
  Model       : meta-llama/Llama-3.1-8B-Instruct
  Steer layer : 20
  Steer alpha : 13.0
  Cache dir   : .style_cache
  Contrast pairs: empathetic=16, formal=16
✓ spaCy en_core_web_sm loaded for KG-NER


In [17]:
# ─── Cell 6 — Load Llama-3.1-8B-Instruct (GPU-first, no CPU offload) ──────────
#
# IMPORTANT: main.py's _get_model_and_tokenizer() loads WITHOUT device_map,
# so all layers default to CPU or are split inconsistently under memory pressure.
# This cell loads the model with device_map={'': 0} (all layers → GPU 0) and
# injects it into main_module._model_cache so build_style_vector() never reloads.
#
# Llama-3.1-8B-Instruct needs ~16 GB VRAM in bfloat16.
# Requires an A100 (40 GB) or L4 (24 GB) runtime.

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = os.environ['LOCAL_MODEL_NAME']  # meta-llama/Llama-3.1-8B-Instruct
HF_TOKEN   = os.environ['HF_TOKEN']

MODEL_NEEDS_GB = 16.5   # Llama-3.1-8B in bfloat16

# ── Choose device strategy ──────────────────────────────────────────────────
if torch.cuda.is_available():
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    free_vram_gb  = (torch.cuda.get_device_properties(0).total_memory
                     - torch.cuda.memory_reserved()) / 1e9
    print(f'  GPU total VRAM  : {total_vram_gb:.1f} GB')
    print(f'  VRAM free now   : {free_vram_gb:.1f} GB')
else:
    total_vram_gb = free_vram_gb = 0

if free_vram_gb >= MODEL_NEEDS_GB:
    device_map_arg = {'': 0}   # all layers → GPU 0, zero CPU offload
    max_memory_arg = None
    print('  Strategy        : GPU-only (device_map={"":0}) — no CPU offload')
elif total_vram_gb >= MODEL_NEEDS_GB:
    print('  ⚠  Free VRAM tight — total capacity is sufficient.')
    print('  Re-run Cell 0a to free reserved memory, then retry.')
    gpu_limit      = f'{int(total_vram_gb - 2)}GiB'
    device_map_arg = 'auto'
    max_memory_arg = {0: gpu_limit, 'cpu': '4GiB'}
    print(f'  Strategy        : auto + max_memory={{GPU: {gpu_limit}, cpu: 4GiB}}')
else:
    print(f'  ⚠  ERROR: Only {total_vram_gb:.1f} GB total VRAM — need ~{MODEL_NEEDS_GB} GB.')
    print('  Upgrade: Runtime → Change runtime type → A100 (40 GB) or L4 (24 GB)')
    device_map_arg = 'auto'
    max_memory_arg = None

print(f'\nLoading {MODEL_NAME} ...')
print('(~16 GB download on first run; ~1-2 min from Drive cache on later runs)')

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    token=HF_TOKEN,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

_load_kwargs = dict(
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map=device_map_arg,
    token=HF_TOKEN,
)
if max_memory_arg is not None:
    _load_kwargs['max_memory'] = max_memory_arg

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **_load_kwargs)
model.eval()

# Inject into main_module cache so build_style_vector() reuses this instance
main_module._model_cache['model']     = model
main_module._model_cache['tokenizer'] = tokenizer

# ── Post-load report ────────────────────────────────────────────────────────
if torch.cuda.is_available():
    vram_used = torch.cuda.memory_allocated() / 1e9
    vram_res  = torch.cuda.memory_reserved()  / 1e9
    print()
    print(f'✓ {MODEL_NAME} loaded')
    print(f'  VRAM allocated  : {vram_used:.2f} GB')
    print(f'  VRAM reserved   : {vram_res:.2f} GB')
    cpu_params = sum(1 for p in model.parameters() if p.device.type in ('cpu','meta'))
    gpu_params = sum(1 for p in model.parameters() if p.device.type == 'cuda')
    print(f'  GPU params      : {gpu_params}')
    if cpu_params > 0:
        print(f'  ⚠  CPU/meta params: {cpu_params} — some layers offloaded to CPU!')
        print('  Restart runtime, run Cell 0a, and retry if this causes errors.')
    else:
        print('  ✓ All parameters on GPU — no CPU offload!')

n_layers = len(model.model.layers) if hasattr(model,'model') and hasattr(model.model,'layers') else '?'
hidden   = model.config.hidden_size if hasattr(model,'config') else '?'
print(f'  Layers          : {n_layers}  (STEER_LAYER={os.environ["STEER_LAYER"]})')
print(f'  Hidden dim      : {hidden}  (expected 4096 for 8B)')

# Verify Llama-3.1 EOS token IDs
_eos_ids = [tokenizer.eos_token_id]
_eot_id  = tokenizer.convert_tokens_to_ids('<|eot_id|>')
if _eot_id and _eot_id not in _eos_ids:
    _eos_ids.append(_eot_id)
print(f'  EOS token IDs   : {_eos_ids}  (should include both 128001 + 128009)')
print(f'  _model_cache    : populated ✓')


  GPU total VRAM  : 50.9 GB
  VRAM free now   : 40.1 GB
  Strategy        : GPU-only (device_map={"":0}) — no CPU offload

Loading meta-llama/Llama-3.1-8B-Instruct ...
(~16 GB download on first run; ~1-2 min from Drive cache on later runs)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


✓ meta-llama/Llama-3.1-8B-Instruct loaded
  VRAM allocated  : 26.71 GB
  VRAM reserved   : 26.94 GB
  GPU params      : 291
  ✓ All parameters on GPU — no CPU offload!
  Layers          : 32  (STEER_LAYER=20)
  Hidden dim      : 4096  (expected 4096 for 8B)
  EOS token IDs   : [128009]  (should include both 128001 + 128009)
  _model_cache    : populated ✓


In [18]:
# ─── Cell 7 — Build EMPATHETIC style vector ───────────────────────────────────
# Model is already loaded in Cell 6 and cached — no re-download.
# Expected time: ~2-4 min on A100 (16 contrast pairs × 2 forward passes each)

print('Building EMPATHETIC vector (16 contrast pairs × 2 activations)...')
print('Expected time: ~2-4 min on A100')
print()

vec_emp = main_module.build_style_vector(style='empathetic', method='pca')

print(f'\n✓ EMPATHETIC vector built')
print(f'  shape : {list(vec_emp.shape)}')
print(f'  norm  : {vec_emp.norm().item():.6f}  (should be ~1.0)')
print(f'  dtype : {vec_emp.dtype}')


We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


Building EMPATHETIC vector (16 contrast pairs × 2 activations)...
Expected time: ~2-4 min on A100


✓ EMPATHETIC vector built
  shape : [4096]
  norm  : 1.000000  (should be ~1.0)
  dtype : torch.float32


In [19]:
# ─── Cell 8 — Build FORMAL style vector ───────────────────────────────────────
# Model already cached from Cell 7 — this cell is fast (~30-60 sec).

print('Building FORMAL vector (model already loaded)...')
print()

vec_form = main_module.build_style_vector(style='formal', method='pca')

print(f'\n✓ FORMAL vector built')
print(f'  shape : {list(vec_form.shape)}')
print(f'  norm  : {vec_form.norm().item():.6f}  (should be ~1.0)')
print(f'  dtype : {vec_form.dtype}')


Building FORMAL vector (model already loaded)...




✓ FORMAL vector built
  shape : [4096]
  norm  : 0.999999  (should be ~1.0)
  dtype : torch.float32


In [20]:
# ─── Cell 9 — Verify both vectors ─────────────────────────────────────────────

import torch

HIDDEN_DIM = 4096   # Llama-3.1-8B hidden dimension (same as Llama-2-7b)

sim = torch.nn.functional.cosine_similarity(
    vec_emp.unsqueeze(0), vec_form.unsqueeze(0)
).item()

print('=' * 55)
print(f'  empathetic : norm={vec_emp.norm():.6f}  shape={list(vec_emp.shape)}')
print(f'  formal     : norm={vec_form.norm():.6f}  shape={list(vec_form.shape)}')
print(f'  cosine sim : {sim:.4f}')
print('=' * 55)

assert abs(vec_emp.norm().item()  - 1.0) < 0.01, f'empathetic not unit-norm! {vec_emp.norm():.4f}'
assert abs(vec_form.norm().item() - 1.0) < 0.01, f'formal not unit-norm! {vec_form.norm():.4f}'
assert vec_emp.shape[0]  == HIDDEN_DIM, f'Expected {HIDDEN_DIM}, got {vec_emp.shape[0]}'
assert vec_form.shape[0] == HIDDEN_DIM, f'Expected {HIDDEN_DIM}, got {vec_form.shape[0]}'
assert sim < 0.5, f'Cosine sim {sim:.3f} too high — vectors may not be distinct!'

print()
if sim < 0:
    print('✓ Cosine similarity NEGATIVE — vectors point in opposite directions (ideal)')
else:
    print(f'  sim={sim:.3f} > 0 — vectors diverge but are not anti-parallel.')
    print('  Acceptable as long as sim < 0.5.')

import os
cache_files = sorted(os.listdir('.style_cache'))
print('\nCached files in .style_cache/:')
for f in cache_files:
    size = os.path.getsize(f'.style_cache/{f}') / 1024
    print(f'  {f}  ({size:.1f} KB)')
print('  Note: ~16 KB each = 4096 float32 values (Llama-3.1-8B hidden dim)')
print()
print('✓ All assertions passed!')


  empathetic : norm=1.000000  shape=[4096]
  formal     : norm=0.999999  shape=[4096]
  cosine sim : -0.9820

✓ Cosine similarity NEGATIVE — vectors point in opposite directions (ideal)

Cached files in .style_cache/:
  style_vec_empathetic_pca.pkl  (16.4 KB)
  style_vec_formal_pca.pkl  (16.4 KB)
  Note: ~16 KB each = 4096 float32 values (Llama-3.1-8B hidden dim)

✓ All assertions passed!


In [21]:
# ─── Cell 10 — Smoke test: activation steering changes tone ───────────────────
#
# Llama-3.1-specific:
#   - Prompt format: <|begin_of_text|> / <|start_header_id|> / <|eot_id|>
#   - add_special_tokens=False — special tokens are embedded in the prompt string
#   - eos_token_id is a LIST: both <|end_of_text|> (128001) and <|eot_id|> (128009)
#   - Hook steers ALL positions (not just last token) — matches main.py

import torch

ALPHA_TEST = 13.0

model, tokenizer = main_module._get_model_and_tokenizer()

# Build dual EOS list (same logic as main.py _run_kg_activation_steering)
_eos_ids = [tokenizer.eos_token_id]
_eot_id  = tokenizer.convert_tokens_to_ids('<|eot_id|>')
if _eot_id and _eot_id not in _eos_ids:
    _eos_ids.append(_eot_id)
print(f'EOS token IDs : {_eos_ids}  (128001=<|end_of_text|>, 128009=<|eot_id|>)')

# Llama-3.1-Instruct prompt format
system_msg = (
    'You are a professional customer support agent. '
    'Write a support reply. Do NOT use headers or bullet points. '
    'Write ONLY the reply — 3 to 4 sentences maximum.'
)
test_prompt = (
    '<|begin_of_text|>'
    '<|start_header_id|>system<|end_header_id|>\n\n'
    f'{system_msg}<|eot_id|>'
    '<|start_header_id|>user<|end_header_id|>\n\n'
    'Customer message: My <PRODUCT> order <ORDER_ID> has an <ISSUE>.<|eot_id|>'
    '<|start_header_id|>assistant<|end_header_id|>\n\n'
)

def generate_steered(prompt, style_vec, alpha, label):
    device = next(model.parameters()).device
    vec    = style_vec.to(device=device, dtype=torch.float32)
    layer  = main_module._get_layer(model)

    def _hook(module, inp, output):
        h = output[0] if isinstance(output, tuple) else output
        h_f32 = h.float() + alpha * vec   # steer ALL positions
        steered = h_f32.to(h.dtype)
        return (steered,) + output[1:] if isinstance(output, tuple) else steered

    handle = layer.register_forward_hook(_hook)
    # add_special_tokens=False — special tokens already embedded in prompt string
    inputs = tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to(device)
    try:
        with torch.no_grad():
            out = model.generate(
                inputs.input_ids,
                max_new_tokens=80,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.3,
                eos_token_id=_eos_ids,   # both EOS tokens required for Llama-3.1
            )
    finally:
        handle.remove()

    new_ids = out[0][inputs.input_ids.shape[1]:]
    text    = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    print(f'\n[{label}]\n  {text[:300]}')
    return text

print('=' * 60)
print('Smoke Test: same prompt, two steering vectors, different tones')
print('=' * 60)

r_emp  = generate_steered(test_prompt, vec_emp,  ALPHA_TEST, 'EMPATHETIC')
r_form = generate_steered(test_prompt, vec_form, ALPHA_TEST, 'FORMAL')

print()
print('=' * 60)
print('✓ Smoke test complete')
print('  EMPATHETIC should use warm personal language ("I\'m so sorry", "I understand")')
print('  FORMAL should use institutional language ("We acknowledge", "has been logged")')
print('=' * 60)


/home/pakdd/.venv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/pakdd/.venv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask is not set and cannot be inferred from input because pad token is same as eos token.As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


EOS token IDs : [128009]  (128001=<|end_of_text|>, 128009=<|eot_id|>)
Smoke Test: same prompt, two steering vectors, different tones

[EMPATHETIC]
  We apologize for any inconvenience caused by this issue and appreciate your reference number (Order ID) for further assistance. Please allow us time to investigate the matter, as we will contact you via email once additional information is required from our end. A resolution update will be provided 

[FORMAL]
  I'd love to help you with that! Can you tell me exactly what's going on and how I can make it right for you? What do you mean by "<ISSUE>" - is there something wrong with your delivery, like where was my package sent instead of being held at home just got delivered yesterday but one thing missing...

✓ Smoke test complete
  EMPATHETIC should use warm personal language ("I'm so sorry", "I understand")
  FORMAL should use institutional language ("We acknowledge", "has been logged")


In [22]:
# ─── Cell 11 — Verify saved .pkl files and next steps ─────────────────────────

import os, pickle, torch
from pathlib import Path

cache_dir = Path('.style_cache')
pkl_files = sorted(cache_dir.glob('*.pkl'))

print('=' * 60)
print('Stored vectors in .style_cache/')
print('=' * 60)

if not pkl_files:
    print('  (no .pkl files found — did Cells 7 and 8 run successfully?)')
else:
    for pkl_path in pkl_files:
        size_kb = pkl_path.stat().st_size / 1024
        with open(pkl_path, 'rb') as f:
            v = pickle.load(f)
        print(f'  {pkl_path.name:<45} {size_kb:6.2f} KB  '
              f'shape={list(v.shape)}  norm={v.norm():.4f}')

print('=' * 60)
print(f'Cache dir (absolute): {cache_dir.resolve()}')
print()
print('  Each file is ~16 KB — 4096 float32 values (Llama-3.1-8B hidden dim)')
print()
print('Next steps:')
print('  1. Download the .style_cache/ folder')
print('  2. Place it next to main.py')
print('  3. Set in .env:')
print('       LOCAL_MODEL_NAME=meta-llama/Llama-3.1-8B-Instruct')
print('       STEER_LAYER=20')
print('       STEER_ALPHA=13.0')
print('       STYLE_CACHE_DIR=.style_cache')
print('       HF_TOKEN=hf_...')
print('  4. Run: python main.py battery_issue')
print('     Or:  python main.py batch')
print()
print('✓ Vector build pipeline complete!')


Stored vectors in .style_cache/
  style_vec_empathetic_pca.pkl                   16.39 KB  shape=[4096]  norm=1.0000
  style_vec_formal_pca.pkl                       16.39 KB  shape=[4096]  norm=1.0000
Cache dir (absolute): /home/pakdd/github repo/EMNLP 2026/A2A_KG_Llama-3.1-8b-Instruct/.style_cache

  Each file is ~16 KB — 4096 float32 values (Llama-3.1-8B hidden dim)

Next steps:
  1. Download the .style_cache/ folder
  2. Place it next to main.py
  3. Set in .env:
       LOCAL_MODEL_NAME=meta-llama/Llama-3.1-8B-Instruct
       STEER_LAYER=20
       STEER_ALPHA=13.0
       STYLE_CACHE_DIR=.style_cache
       HF_TOKEN=hf_...
  4. Run: python main.py battery_issue
     Or:  python main.py batch

✓ Vector build pipeline complete!


In [23]:
# ─── Cell 13 — [Optional] Run the full A2A pipeline (single scenario) ────────
# Requires GROQ_API_KEY.
# Scenarios: battery_issue | wrong_item | billing_error | delivery_delay
import asyncio

SCENARIO = "battery_issue"   # ← change to try other scenarios

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY or GROQ_API_KEY == "gsk_YOUR_KEY_HERE":
    print("⚠  Skipping: GROQ_API_KEY not set. Fill it in Cell 2.")
else:
    from langchain_groq import ChatGroq
    llm          = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)
    llm_customer = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)
    try:
        import nest_asyncio; nest_asyncio.apply()
    except ImportError:
        import subprocess; subprocess.run(["pip", "install", "-q", "nest_asyncio"], check=True)
        import nest_asyncio; nest_asyncio.apply()
    print(f"Running pipeline — scenario: {SCENARIO}")
    record = asyncio.run(main_module.run_pipeline(SCENARIO, llm, llm_customer))
    emp_out = {"kg_summary": record["knowledge_graph"],
               "defactualized_prompt": record["empathetic_output"]["defactualized_prompt"]}
    main_module.console.print(main_module.make_kg_panel(emp_out))
    emp_response  = {**record["empathetic_output"], "kg_summary": record["knowledge_graph"]}
    form_response = {**record["formal_output"],     "kg_summary": record["knowledge_graph"]}
    main_module.console.print(main_module.make_comparison_table(emp_response, form_response))
    print(f"\nFull record keys: {list(record.keys())}")


Running pipeline — scenario: battery_issue


/home/pakdd/.venv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/pakdd/.venv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


╭─────────────────────────────────────────────── Customer Message ────────────────────────────────────────────────╮
│ Customer: Priya Sharma   Order: ORD-7741                                                                        │
│ Product: TechPro X200 Laptop                                                                                    │
│ Issue: battery draining completely within 2 hours of a full charge                                              │
│ Sentiment: FRUSTRATED   Urgency: high                                                                           │
│                                                                                                                 │
│ "I'm reaching out again regarding my TechPro X200 Laptop, order number ORD-7741, which I purchased just three   │
│ weeks ago. As I mentioned last week, the battery is draining extremely quickly - it's completely dead within 2  │
│ hours of a full charge, which is making it impossible for me to meet my deadlines. I did update the drivers as  │
│ suggested, but unfortunately, that didn't resolve the issue. With a critical deadline looming for tomorrow, I   │
│ urgently need a more effective solution to this problem today."                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────── Empathetic — Support Response ───────────────────╮
│ Style: EMPATHETIC   Steering mode: kg_activation                   │
│                                                                    │
│ Final response (rehydrated):                                       │
│ Given your urgent situation with an upcoming deadline, we're going │
│ to expedite our troubleshooting process by sending over one of our │
│ technical specialists who will work closely with you directly via  │
│ phone call to identify the root cause of the issue and provide a   │
│ suitable resolution before tomorrow morning. Please expect their   │
│ contact information shortly so they can assist you further         │
│ immediately upon receipt. In addition, please note that any        │
│ necessary replacement parts or services related to this matter may │
│ also be provided at no additional cost if required.                │
│                                                                    │
│ Defactualized output (pre-rehydration):                            │
│ Given your urgent situation with an upcoming deadline, we're going │
│ to expedite our troubleshooting process by sending over one of our │
│ technical specialists who will work closely with you directly via  │
╰────────────────────────────────────────────────────────────────────╯
╭──────────────────── Formal — Support Response ─────────────────────╮
│ Style: FORMAL   Steering mode: kg_activation                       │
│                                                                    │
│ Final response (rehydrated):                                       │
│ A replacement unit with an upgraded power management system will   │
│ be expedited via overnight shipping in accordance with our         │
│ warranty policy. You should receive tracking information shortly   │
│ after processing by end-of-day today at the latest. Please contact │
│ us if any further assistance is required prior to receiving the    │
│ new device.                                                        │
│                                                                    │
│ Defactualized output (pre-rehydration):                            │
│ A replacement unit with an upgraded power management system will   │
│ be expedited via overnight shipping in accordance with our         │
│ warranty policy. You should receive tracking information shortly   │
│ after proce                                                        │
╰────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Knowledge Graph ────────────────────────────────────────────────╮
│ Nodes (6):                                                                                                      │
│   (PRODUCT) TechPro X200 Laptop  (salience: 0.8)                                                                │
│   (ORDER_ID) ORD-7741  (salience: 0.9)                                                                          │
│   (URGENCY) high  (salience: 1.0)                                                                               │
│   (SENTIMENT) neutral  (salience: 0.6)                                                                          │
│   (CUSTOMER_NAME) Priya Sharma  (salience: 0.7)                                                                 │
│   (ISSUE) battery draining completely within 2 hours of a full charge  (salience: 0.7)                          │
│                                                                                                                 │
│ Edges (4):                                                                                                      │
│   TechPro X200 Laptop       —has_issue→ battery draining complete                                               │
│   ORD-7741                  —about_product→ TechPro X200 Laptop                                                 │
│   Priya Sharma              —placed_order→ ORD-7741                                                             │
│   battery draining complete —has_urgency→ high                                                                  │
│                                                                                                                 │
│ Defactualized prompt:                                                                                           │
│   I'm reaching out again regarding my <PRODUCT>, order number <ORDER_ID>, which I purchased just three weeks    │
│ ago. As I mentioned last week, the battery is draining extremely quickly - it's completely de                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                   Style Comparison — Same Facts, Different Tone                                   
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Dimension       ┃ Empathetic Pipeline                           ┃ Formal Pipeline                               ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Target style    │ empathetic                                    │ formal                                        │
│ Steering mode   │ kg_activation                                 │ kg_activation                                 │
│ KG nodes        │ 6                                             │ 6                                             │
│ KG edges        │ 4                                             │ 4                                             │
│ Response (first │ Given your urgent situation with an upcoming  │ A replacement unit with an upgraded power     │
│ 200 chars)      │ deadline, we're going to expedite our         │ management system will be expedited via       │
│                 │ troubleshooting process by sending over one   │ overnight shipping in accordance with our     │
│                 │ of our technical specialists who will work    │ warranty policy. You should receive tracking  │
│                 │ closely with you directly via                 │ information shortly after proce               │
└─────────────────┴───────────────────────────────────────────────┴───────────────────────────────────────────────┘


Full record keys: ['input', 'knowledge_graph', 'empathetic_output', 'formal_output']


In [24]:
# ─── Cell 14 — [Optional] Run the full 100-case batch ───────────────────────
# ⚠  ~15–40 min depending on Groq rate limits.
import asyncio
from pathlib import Path

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY or GROQ_API_KEY == "gsk_YOUR_KEY_HERE":
    print("⚠  Skipping: GROQ_API_KEY not set.")
else:
    Path("outputs_3.1_8b").mkdir(exist_ok=True)
    Path("outputs_3.1_8b/knowledge_graphs").mkdir(exist_ok=True)
    try:
        import nest_asyncio; nest_asyncio.apply()
    except ImportError:
        import subprocess; subprocess.run(["pip", "install", "-q", "nest_asyncio"], check=True)
        import nest_asyncio; nest_asyncio.apply()
    print(f"Starting batch — {len(main_module.BATCH_CASES)} cases")
    asyncio.run(main_module.run_batch())


Starting batch — 100 cases


──────────────────────────────────── A2A Customer Support — 100-Case Batch Run ────────────────────────────────────

Output file : outputs/results_20260524_045623.jsonl
  Total cases : 100
  Activation steering: ENABLED

──────────────── Case   1/100  scenario=battery_issue  customer=Priya Sharma  sentiment=frustrated ────────────────

╭─────────────────────────────────────────────── Customer Message ────────────────────────────────────────────────╮
│ Customer: Priya Sharma   Order: ORD-7741                                                                        │
│ Product: TechPro X200 Laptop                                                                                    │
│ Issue: battery draining completely within 2 hours of a full charge                                              │
│ Sentiment: FRUSTRATED   Urgency: high                                                                           │
│                                                                                                                 │
│ "I'm reaching out to support for the second time regarding my TechPro X200 Laptop, which I purchased just three │
│ weeks ago (order number ORD-7741). Despite updating the drivers as suggested during my previous support call,   │
│ the battery issue persists - it's still draining completely within 2 hours of a full charge. I'm on a tight     │
│ deadline with a client project that's due tomorrow, so I urgently need a more effective solution to this        │
│ problem. Could you please assist me in resolving this issue today?"                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── Customer Message ────────────────────────────────────────────────╮
│ Customer: Ximena Flores   Order: ORD-8858                                                                       │
│ Product: Ergonomic Office Chair                                                                                 │
│ Issue: delivery now 3 weeks overdue with no update from courier                                                 │
│ Sentiment: NEUTRAL   Urgency: medium                                                                            │
│                                                                                                                 │
│ "I'm reaching out about my order, ORD-8858, for the Ergonomic Office Chair, which was supposed to arrive three  │
│ weeks ago. Unfortunately, I still haven't received any updates from the courier, and I'm starting to get        │
│ concerned that it's been delayed indefinitely. I understand that delays can happen, but I would appreciate it   │
│ if you could provide me with a clear timeline for when I can expect the chair to arrive, or let me know if I'm  │
│ eligible for a refund so I can explore other options locally. Could you please look into this and get back to   │
│ me as soon as possible?"                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────── Empathetic — Support Response ───────────────────╮
│ Style: EMPATHETIC   Steering mode: kg_activation                   │
│                                                                    │
│ Final response (rehydrated):                                       │
│ Please rest assured that we're taking immediate action on your     │
│ case - our team will investigate further and reach out directly    │
│ via email within the next two business days regarding an estimated │
│ delivery date or alternative arrangements such as a full refund.   │
│ In the meantime, feel free to contact us at [support phone number] │
│ should you need assistance sooner than expected. We value your     │
│ patience during these circumstances.                               │
│                                                                    │
│ Defactualized output (pre-rehydration):                            │
│ Please rest assured that we're taking immediate action on your     │
│ case - our team will investigate further and reach out directly    │
│ via email within the next two business days regarding an estimated │
│ delive                                                             │
╰────────────────────────────────────────────────────────────────────╯
╭──────────────────── Formal — Support Response ─────────────────────╮
│ Style: FORMAL   Steering mode: kg_activation                       │
│                                                                    │
│ Final response (rehydrated):                                       │
│ Our team will investigate further regarding the status of your     │
│ shipment and communicate directly with our logistics partner to    │
│ obtain an updated delivery estimate. We aim to respond within two  │
│ business days with either a revised arrival date or information on │
│ potential alternatives such as refunds or store credit exchanges.  │
│ In the meantime, we recommend checking with local retailers in     │
│ case they have similar products available for immediate purchase.  │
│                                                                    │
│ Defactualized output (pre-rehydration):                            │
│ Our team will investigate further regarding the status of your     │
│ shipment and communicate directly with our logistics partner to    │
│ obtain an updated delivery estimate. We aim to respond within two  │
│ busines                                                            │
╰────────────────────────────────────────────────────────────────────╯

✓ Saved case 100  |  KG → knowledge_graphs/kg_100_delivery_delay_ORD-8858_Ximena_Flores.json

───────────────────────────────────────────────── Batch complete ──────────────────────────────────────────────────

Success: 100/100
  Failed:  0/100
  Results: outputs/results_20260524_045623.jsonl
  KGs:     outputs/knowledge_graphs/  (100 files)